# Step 4 — Map Match Traffic → OSM Edges

This notebook assigns real-time Mapbox traffic congestion to every edge of the duckOSM
road network and writes the result back into the DuckDB database.

## What it does

1. **Reads edges** from `driving.edges` in the DuckDB (produced by notebook 2)
2. **Reads traffic segments** from the GeoJSON produced by notebook 3
3. **Matches** each traffic segment to all OSM edges it covers using:
   - R-tree spatial index for fast candidate selection (25 m corridor)
   - Direction filter — rejects parallel / opposite roads (< 45°)
   - Overlap filter — at least 40% of the OSM edge must lie in the corridor
4. **Writes congestion back** into `driving.edges` in the DuckDB
5. **Exports** enriched CSV and GeoJSON
6. **Reports statistics** — match rate, breakdown by road type and congestion level

## Algorithm: one-to-many geometric matching

One Mapbox traffic segment typically covers **many** OSM edges (it is longer).  
For each traffic segment `T`, we find **all** OSM edges `E` it covers — not just the nearest one.  
If an edge is matched by multiple segments, the most severe congestion wins:
`severe > heavy > moderate > low`

```
boundary corridor (25 m)
     ┌──────────────┐
     │  ████████████│  traffic segment T  (congestion: low)
     │  ─────────── │  OSM edge E₁  ✓ (75% overlap, direction OK)
     │  ─────── ────│  OSM edge E₂  ✗ (perpendicular, rejected)
     │  ────────────│  OSM edge E₃  ✓ (100% overlap, direction OK)
     └──────────────┘
```

In [ ]:
%%time
import duckdb
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import matplotlib.pyplot as plt
from pathlib import Path
from shapely import get_coordinates

# ── Configuration ─────────────────────────────────────────────────────────
NAME       = 'sodermalm'
DB_PATH    = Path(f'../db/{NAME}.duckdb')
TRAFFIC_FILE = Path(f'../output/{NAME}_traffic.geojson')  # from notebook 3
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Matching parameters (tune if needed)
BUF_M         = 25     # corridor width in metres
DIR_THRESH    = 45     # max bearing difference in degrees
OVERLAP_THRESH = 0.40  # min fraction of OSM edge length that must be inside corridor
# ─────────────────────────────────────────────────────────────────────────

print(f'DuckDB      : {DB_PATH}')
print(f'Traffic     : {TRAFFIC_FILE}')
print(f'Parameters  : buf={BUF_M}m  dir<{DIR_THRESH}°  overlap≥{OVERLAP_THRESH*100:.0f}%')

---
## Step 0 — Load data

**OSM edges** are read directly from the duckOSM DuckDB database.  
The geometry is stored as a DuckDB `GEOMETRY` type — we export it as WKT and parse into shapely.

**Mapbox traffic segments** come from the GeoJSON produced by notebook 3.  
We keep only line geometries (not points like traffic signals) and only segments
with real congestion data (not `no data`).

In [ ]:
%%time
# ── Read OSM edges from DuckDB ─────────────────────────────────────────────
con = duckdb.connect(str(DB_PATH))
con.execute('LOAD spatial')

df = con.execute("""
    SELECT edge_id, highway, name, oneway, length_m,
           ST_AsText(geometry) AS wkt_geom
    FROM driving.edges
""").df()

edges = gpd.GeoDataFrame(
    df,
    geometry=gpd.GeoSeries.from_wkt(df['wkt_geom']),
    crs='EPSG:4326',
).reset_index(drop=True)

# ── Read Mapbox traffic segments ───────────────────────────────────────────
traffic_all = gpd.read_file(TRAFFIC_FILE)
traffic = traffic_all[
    traffic_all.geometry.geom_type.isin(['LineString', 'MultiLineString'])
    & (traffic_all['congestion'] != 'no data')
].reset_index(drop=True)

print(f'OSM edges         : {len(edges):,}')
print(f'Traffic segments  : {len(traffic):,}  (with congestion)')
print(f'Congestion values : {traffic["congestion"].value_counts().to_dict()}')

### Coordinate projection

Both datasets are re-projected from **WGS-84 (EPSG:4326)** to **Web Mercator (EPSG:3857)** (metres).  
This is required for correct buffer distances and overlap lengths —
`buffer(25)` in degrees would mean ~2 800 km, not 25 m.

In [ ]:
%%time
edges_m   = edges.to_crs('EPSG:3857').reset_index(drop=True)
traffic_m = traffic.to_crs('EPSG:3857').reset_index(drop=True)
print('Projected to EPSG:3857 (metres)')

---
## Step 1 — Helper functions

### `bearing(geom)`
Primary direction of a LineString as degrees [0°, 360°) — computed from start to end point.

### `dir_diff(b1, b2)`
Smallest angle between two bearings (0°–90°). Roads are bidirectional so 10° and 190° are the same road.

### `all_matches(traffic_geom, osm_gdf, osm_sindex)`
For a single traffic segment:
1. **R-tree query** — find OSM edges whose bounding box overlaps the 25 m corridor (fast)
2. **Direction filter** — drop edges whose bearing differs by more than 45° (rejects parallel roads)
3. **Overlap filter** — drop edges where < 40% of their length lies inside the corridor
4. Return **all** passing indices (one-to-many)

In [ ]:
%%time
def bearing(geom):
    coords = get_coordinates(geom)
    dx = coords[-1][0] - coords[0][0]
    dy = coords[-1][1] - coords[0][1]
    return np.degrees(np.arctan2(dx, dy)) % 360

def dir_diff(b1, b2):
    d = abs(b1 - b2) % 360
    return min(d, 180 - d)

def all_matches(traffic_geom, osm_gdf, osm_sindex):
    corridor = traffic_geom.buffer(BUF_M)
    hits = list(osm_sindex.intersection(corridor.bounds))
    if not hits:
        return []
    t_bear = bearing(traffic_geom)
    matched = []
    for idx in hits:
        edge = osm_gdf.iloc[idx].geometry
        if dir_diff(t_bear, bearing(edge)) > DIR_THRESH:
            continue
        overlap = corridor.intersection(edge).length / max(edge.length, 1e-6)
        if overlap >= OVERLAP_THRESH:
            matched.append(idx)
    return matched

print('Helper functions defined')

---
## Step 2 — Run matching (Traffic → OSM, one-to-many)

For every traffic segment with a real congestion value, find all OSM edges it covers
and assign the congestion level. When multiple segments match the same edge, the
most severe value wins: `severe (4) > heavy (3) > moderate (2) > low (1)`.

**Expected output:**
- `Total (T, E) assignments` — total matched pairs (avg >> 1 confirms one-to-many is working)
- `OSM edges with congestion` — should be > 90% for good Mapbox coverage areas

In [ ]:
%%time
SEVERITY = {'severe': 4, 'heavy': 3, 'moderate': 2, 'low': 1}

osm_sindex      = edges_m.sindex
edge_congestion = {}   # edge_id (int) → congestion string
total_pairs     = 0

for _, row in traffic_m.iterrows():
    matched = all_matches(row.geometry, edges_m, osm_sindex)
    cong    = row['congestion']
    for idx in matched:
        edge_id = int(edges_m.iloc[idx]['edge_id'])
        if SEVERITY.get(cong, 0) > SEVERITY.get(edge_congestion.get(edge_id, ''), 0):
            edge_congestion[edge_id] = cong
    total_pairs += len(matched)

edges['congestion'] = edges['edge_id'].map(edge_congestion).fillna('no data')

print(f'Traffic segments processed : {len(traffic_m):,}')
print(f'Total (T, E) assignments   : {total_pairs:,}')
print(f'OSM edges with congestion  : {len(edge_congestion):,} / {len(edges):,}')
print(f'Avg edges per segment      : {total_pairs/max(len(traffic_m),1):.1f}')
print()
print('Congestion breakdown:')
print(edges['congestion'].value_counts().to_string())

---
## Step 3 — Write congestion back to DuckDB

The `congestion` column is added to `driving.edges` in the DuckDB database.  
This enriches the routing network in-place — any downstream tool reading the DuckDB
will see the congestion column alongside all the existing routing data (costs, H3 cells, etc.).

The `ADD COLUMN IF NOT EXISTS` clause makes re-runs safe — if the column already exists
it is simply overwritten.

In [ ]:
%%time
con.execute("ALTER TABLE driving.edges ADD COLUMN IF NOT EXISTS congestion VARCHAR DEFAULT 'no data'")

rows = [(cong, eid) for eid, cong in edge_congestion.items()]
con.executemany("UPDATE driving.edges SET congestion = ? WHERE edge_id = ?", rows)
con.close()

print(f'Congestion written to {DB_PATH}')
print(f'  {len(rows):,} edges updated')
print(f'  Verify: SELECT congestion, count(*) FROM driving.edges GROUP BY 1')

---
## Step 4 — Visualize

Line geometries only — points (traffic signals, stop signs) are excluded.

| Color | Congestion | Meaning |
|---|---|---|
| 🟢 Green | `low` | Traffic flowing freely |
| 🟠 Orange | `moderate` | Slightly reduced speed |
| 🔴 Red | `heavy` | Significant slowdown |
| ⬛ Dark red | `severe` | Near standstill |
| ⬜ Light gray | `no data` | No Mapbox sensor coverage |

In [ ]:
%%time
COLORS = {'low':'#00c800','moderate':'#ffa500','heavy':'#ff4500','severe':'#8b0000','no data':'#cccccc'}

lines  = edges[edges.geometry.geom_type.isin(['LineString','MultiLineString'])]
bds    = lines.total_bounds
center = [(bds[1]+bds[3])/2, (bds[0]+bds[2])/2]

m = folium.Map(location=center, zoom_start=14, tiles='OpenStreetMap')
folium.GeoJson(
    lines.__geo_interface__,
    style_function=lambda feat: {
        'color':  COLORS.get(feat['properties'].get('congestion','no data'), '#cccccc'),
        'weight': 3 if feat['properties'].get('congestion','no data') != 'no data' else 1,
    },
    tooltip=None, popup=None,
).add_to(m)
m

---
## Step 5 — Matching statistics

Quantitative evaluation of match quality:
- **Overall** — edge match rate, length match rate, avg edges per segment
- **By congestion level** — edge count and km per level
- **By highway type** — which road classes were best covered
- **Charts** — bar (edge counts), horizontal bar (match rate by type), pie (km by congestion)

In [ ]:
%%time
edges_m2            = edges.to_crs('EPSG:3857').copy()
edges_m2['length_m_proj'] = edges_m2.geometry.length
edges_m2['congestion']    = edges['congestion'].values

matched   = edges_m2[edges_m2['congestion'] != 'no data']
unmatched = edges_m2[edges_m2['congestion'] == 'no data']
total_km   = edges_m2['length_m_proj'].sum() / 1000
matched_km = matched['length_m_proj'].sum() / 1000

print('═'*54)
print('  MATCHING SUMMARY')
print('═'*54)
print(f'  OSM edges total           : {len(edges):>7,}')
print(f'  Edges matched             : {len(matched):>7,}  ({len(matched)/len(edges)*100:.1f}%)')
print(f'  Edges unmatched           : {len(unmatched):>7,}  ({len(unmatched)/len(edges)*100:.1f}%)')
print(f'  Network length total      : {total_km:>7.1f} km')
print(f'  Matched length            : {matched_km:>7.1f} km  ({matched_km/total_km*100:.1f}%)')
print(f'  Traffic segments used     : {len(traffic_m):>7,}')
print(f'  Total (T, E) assignments  : {total_pairs:>7,}')
print(f'  Avg edges per segment     : {total_pairs/max(len(traffic_m),1):>7.1f}')
print('═'*54)

print('\nCongestion distribution (matched edges):')
cong_lens = matched.groupby('congestion')['length_m_proj'].sum() / 1000
for level in ['severe','heavy','moderate','low']:
    n  = (matched['congestion'] == level).sum()
    km = cong_lens.get(level, 0.0)
    print(f'  {level:<10}: {n:>5,} edges  {km:>6.1f} km')

print('\nMatch rate by highway type:')
hw_stats = (
    edges_m2.groupby('highway')
    .agg(total=('length_m_proj','count'),
         matched=('congestion', lambda x: (x!='no data').sum()),
         km=('length_m_proj', lambda x: x.sum()/1000))
    .assign(pct=lambda d: d['matched']/d['total']*100)
    .sort_values('pct', ascending=False).reset_index()
)
for _, r in hw_stats.iterrows():
    bar = '█' * int(r['pct']/5)
    print(f"  {r['highway']:<20} {r['pct']:>5.1f}%  {bar:<20}  ({int(r['matched'])}/{int(r['total'])})")

In [ ]:
%%time
CONG_COLORS = {'low':'#00c800','moderate':'#ffa500','heavy':'#ff4500','severe':'#8b0000','no data':'#cccccc'}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Map Matching Statistics — {NAME}', fontsize=13, fontweight='bold')

ax = axes[0]
cong_order = ['low','moderate','heavy','severe','no data']
counts = [edges['congestion'].value_counts().get(c,0) for c in cong_order]
bars = ax.bar(cong_order, counts, color=[CONG_COLORS[c] for c in cong_order], edgecolor='white')
ax.bar_label(bars, fmt='%d', fontsize=8, padding=3)
ax.set_title('Edges by congestion level'); ax.set_xlabel('Congestion'); ax.set_ylabel('OSM edges')
ax.tick_params(axis='x', rotation=15)

ax = axes[1]
top = hw_stats.head(10)
h_bars = ax.barh(top['highway'], top['pct'],
                  color=['#1565C0' if p>=80 else '#42A5F5' for p in top['pct']])
ax.axvline(50, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_title('Match rate by highway type'); ax.set_xlabel('Match rate (%)')
ax.set_xlim(0, 110); ax.bar_label(h_bars, fmt='%.0f%%', fontsize=8, padding=3)

ax = axes[2]
km_by_cong = (matched.groupby('congestion')['length_m_proj'].sum()/1000
              ).reindex(['low','moderate','heavy','severe']).dropna()
ax.pie(km_by_cong.values, labels=km_by_cong.index,
       colors=[CONG_COLORS[l] for l in km_by_cong.index],
       autopct='%1.1f%%', startangle=140, textprops={'fontsize': 9})
ax.set_title(f'Matched road km by congestion\n(total {matched_km:.1f} km)')

plt.tight_layout(); plt.show()

---
## Step 6 — Save output

Two convenience export formats alongside the DuckDB:

| File | Format | Use with |
|---|---|---|
| `{name}_edges_traffic.geojson` | GeoJSON | QGIS, kepler.gl, Mapbox Studio |
| `{name}_edges_traffic.csv` | CSV | pandas, Excel, further analysis |

In [ ]:
%%time
geojson_out = OUTPUT_DIR / f'{NAME}_edges_traffic.geojson'
csv_out     = OUTPUT_DIR / f'{NAME}_edges_traffic.csv'

edges.to_file(geojson_out, driver='GeoJSON')
print(f'Saved GeoJSON → {geojson_out}  ({len(edges):,} edges)')

# CSV — add start/end/centroid coords and segment length
csv_df = edges.drop(columns=['geometry','wkt_geom'], errors='ignore').copy()
clipped_m = edges.to_crs('EPSG:3857')
centroids        = clipped_m.geometry.centroid.to_crs('EPSG:4326')
csv_df['lon']    = centroids.x.round(6)
csv_df['lat']    = centroids.y.round(6)
csv_df['length_m'] = clipped_m.geometry.length.round(1)
all_coords           = edges.geometry.apply(get_coordinates)
csv_df['start_lon']  = all_coords.apply(lambda c: round(c[ 0][0], 6))
csv_df['start_lat']  = all_coords.apply(lambda c: round(c[ 0][1], 6))
csv_df['end_lon']    = all_coords.apply(lambda c: round(c[-1][0], 6))
csv_df['end_lat']    = all_coords.apply(lambda c: round(c[-1][1], 6))
csv_df.to_csv(csv_out, index=False)
print(f'Saved CSV     → {csv_out}')
print()
display(csv_df[['edge_id','highway','name','congestion','length_m','lon','lat']].head(10))